In [34]:
import pandas as pd
import numpy as np

from catboost import CatBoostRegressor

from sklearn.model_selection import train_test_split, GridSearchCV

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.preprocessing import (
    OneHotEncoder,
    OrdinalEncoder
)

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)


df = pd.read_csv("../data/gym_members_exercise_tracking.csv")
df.head()


,Age,Gender,Weight (kg),Height (m),Max_BPM,Avg_BPM,Resting_BPM,Session_Duration (hours),Calories_Burned,Workout_Type,Fat_Percentage,Water_Intake (liters),Workout_Frequency (days/week),Experience_Level,BMI
0,56,Male,88.3,1.71,180,157,60,1.69,1313.0,Yoga,12.6,3.5,4,3,30.20
1,46,Female,74.9,1.53,179,151,66,1.30,883.0,HIIT,33.9,2.1,4,2,32.00
2,32,Female,68.1,1.66,167,122,54,1.11,677.0,Cardio,33.4,2.3,4,2,24.71
3,25,Male,53.2,1.70,190,164,56,0.59,532.0,Strength,28.8,2.1,3,1,18.41
4,38,Male,46.1,1.79,188,158,68,0.64,556.0,Strength,29.2,2.8,3,1,14.39


In [35]:




df.columns = df.columns.str.strip()


print("Dataset shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())


# ============================================================
# 2. DEFINE TARGET
# ============================================================

target = "Calories_Burned"


# ============================================================
# 3. DEFINE FEATURE TYPES
# ============================================================

ordinal_features = [
    "Experience_Level"
]

nominal_features = [
    "Gender",
    "Workout_Type"
]


# IMPORTANT:
# Experience_Level is numeric in the CSV, but we want to
# treat it as ordinal, so explicitly remove it here.

numeric_features = [
    column
    for column in df.select_dtypes(include="number").columns
    if column not in [target] + ordinal_features
]


# ============================================================
# 4. COMBINE FEATURES
# ============================================================

feature_columns = (
    numeric_features
    + ordinal_features
    + nominal_features
)


# ============================================================
# 5. CHECK FOR DUPLICATES
# ============================================================

duplicate_features = (
    pd.Series(feature_columns)
    [pd.Series(feature_columns).duplicated()]
    .tolist()
)

print("\nNumeric features:")
print(numeric_features)

print("\nOrdinal features:")
print(ordinal_features)

print("\nNominal features:")
print(nominal_features)

print("\nDuplicate features:")
print(duplicate_features)


if duplicate_features:
    raise ValueError(
        f"Duplicate features found: {duplicate_features}"
    )


# ============================================================
# 6. CHECK FOR MISSING COLUMNS
# ============================================================

missing_columns = [
    column
    for column in feature_columns + [target]
    if column not in df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing columns: {missing_columns}"
    )


# ============================================================
# 7. CREATE X AND y
# ============================================================

X = df[feature_columns].copy()

y = df[target].copy()


# ============================================================
# 8. REMOVE MISSING VALUES
# ============================================================

data = pd.concat(
    [X, y],
    axis=1
).dropna()

X = data[feature_columns]

y = data[target]


print("\nShape after removing missing values:")
print(X.shape)


# ============================================================
# 9. TRAIN / TEST SPLIT
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)


# ============================================================
# 10. PREPROCESSING
# ============================================================

preprocessor = ColumnTransformer(
    transformers=[

        # Numeric columns
        (
            "numeric",
            "passthrough",
            numeric_features
        ),

        # Ordinal column
        (
            "ordinal",
            OrdinalEncoder(
                handle_unknown="use_encoded_value",
                unknown_value=-1
            ),
            ordinal_features
        ),

        # Nominal columns
        (
            "nominal",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            ),
            nominal_features
        )
    ]
)


# ============================================================
# 11. CATBOOST MODEL
# ============================================================

catboost_model = CatBoostRegressor(
    loss_function="RMSE",
    random_seed=42,
    verbose=False,
    allow_writing_files=False,
    thread_count=2
)


# ============================================================
# 12. PIPELINE
# ============================================================

model_pipeline = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor
        ),

        (
            "model",
            catboost_model
        )
    ]
)


# ============================================================
# 13. HYPERPARAMETER GRID
# ============================================================

param_grid = {

    "model__depth": [
        4,
        6,
        8
    ],

    "model__learning_rate": [
        0.03,
        0.05,
        0.1
    ],

    "model__iterations": [
        300,
        600,
        1000
    ],

    "model__l2_leaf_reg": [
        1,
        3,
        5,
        7
    ]
}


# ============================================================
# 14. MULTIPLE SCORING METRICS
# ============================================================

scoring = {
    "RMSE": "neg_root_mean_squared_error",
    "R2": "r2"
}


# ============================================================
# 15. GRID SEARCH
# ============================================================

tuner = GridSearchCV(
    estimator=model_pipeline,
    param_grid=param_grid,

    # Select best parameters according to RMSE
    scoring=scoring,
    refit="RMSE",

    # 5-fold cross-validation
    cv=5,

    # Use one process because CatBoost is using threads
    n_jobs=1,

    verbose=1
)


# ============================================================
# 16. TRAIN
# ============================================================

print("\nStarting hyperparameter tuning...")

tuner.fit(
    X_train,
    y_train
)


# ============================================================
# 17. BEST MODEL
# ============================================================

best_model = tuner.best_estimator_


# ============================================================
# 18. BEST PARAMETERS
# ============================================================

print("\n")
print("=" * 70)
print("BEST PARAMETERS")
print("=" * 70)

print(tuner.best_params_)


# ============================================================
# 19. CROSS-VALIDATION RESULTS
# ============================================================

best_index = tuner.best_index_

cv_rmse = -tuner.cv_results_[
    "mean_test_RMSE"
][best_index]

cv_r2 = tuner.cv_results_[
    "mean_test_R2"
][best_index]


print("\n")
print("=" * 70)
print("CROSS-VALIDATION RESULTS")
print("=" * 70)

print(f"CV RMSE: {cv_rmse:.3f}")
print(f"CV R²:   {cv_r2:.3f}")


# ============================================================
# 20. TEST SET PREDICTIONS
# ============================================================

predictions = best_model.predict(
    X_test
)


# ============================================================
# 21. TEST METRICS
# ============================================================

test_mae = mean_absolute_error(
    y_test,
    predictions
)

test_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        predictions
    )
)

test_r2 = r2_score(
    y_test,
    predictions
)


# ============================================================
# 22. FINAL RESULTS
# ============================================================

print("\n")
print("=" * 70)
print("TEST SET RESULTS")
print("=" * 70)

print(f"Test MAE:  {test_mae:.3f}")
print(f"Test RMSE: {test_rmse:.3f}")
print(f"Test R²:   {test_r2:.3f}")


# ============================================================
# 23. FEATURE IMPORTANCE
# ============================================================

# Get feature names after preprocessing
feature_names = (
    best_model
    .named_steps["preprocessor"]
    .get_feature_names_out()
)

# Get CatBoost feature importance
importances = (
    best_model
    .named_steps["model"]
    .feature_importances_
)

feature_importance_df = pd.DataFrame({
    "Feature": feature_names,
    "Importance": importances
})

feature_importance_df = (
    feature_importance_df
    .sort_values(
        by="Importance",
        ascending=False
    )
    .reset_index(drop=True)
)


print("\n")
print("=" * 70)
print("FEATURE IMPORTANCE")
print("=" * 70)

display(
    feature_importance_df
)

Dataset shape: (973, 15)

Columns:
['Age', 'Gender', 'Weight (kg)', 'Height (m)', 'Max_BPM', 'Avg_BPM', 'Resting_BPM', 'Session_Duration (hours)', 'Calories_Burned', 'Workout_Type', 'Fat_Percentage', 'Water_Intake (liters)', 'Workout_Frequency (days/week)', 'Experience_Level', 'BMI']

Numeric features:
['Age', 'Weight (kg)', 'Height (m)', 'Max_BPM', 'Avg_BPM', 'Resting_BPM', 'Session_Duration (hours)', 'Fat_Percentage', 'Water_Intake (liters)', 'Workout_Frequency (days/week)', 'BMI']

Ordinal features:
['Experience_Level']

Nominal features:
['Gender', 'Workout_Type']

Duplicate features:
[]

Shape after removing missing values:
(973, 14)

Starting hyperparameter tuning...
Fitting 5 folds for each of 108 candidates, totalling 540 fits


BEST PARAMETERS
{'model__depth': 4, 'model__iterations': 1000, 'model__l2_leaf_reg': 1, 'model__learning_rate': 0.03}


CROSS-VALIDATION RESULTS
CV RMSE: 10.461
CV R²:   0.998


TEST SET RESULTS
Test MAE:  6.645
Test RMSE: 9.253
Test R²:   0.999


FEATU

,Feature,Importance
0,numeric__Session_Duration (hours),60.340988
1,numeric__Avg_BPM,16.609992
2,numeric__Fat_Percentage,7.390690
3,ordinal__Experience_Level,6.183786
4,numeric__Age,4.682917
5,nominal__Gender_Female,1.935919
6,nominal__Gender_Male,1.082233
7,numeric__Workout_Frequency (days/week),0.803416
8,numeric__Water_Intake (liters),0.562417
9,numeric__Weight (kg),0.177445


In [39]:
import joblib

joblib.dump(
    best_model,
    "../models/calories_burned_model.joblib"
)

print("Model saved successfully!")

Model saved successfully!


In [40]:
numeric_cols = [
    "Age",
    "Weight (kg)",
    "Height (m)",
    "Max_BPM",
    "Avg_BPM",
    "Resting_BPM",
    "Session_Duration (hours)",
    "Fat_Percentage",
    "Water_Intake (liters)",
    "Workout_Frequency (days/week)",
    "BMI"
]

print("NUMERIC RANGES")
print("=" * 50)

for col in numeric_cols:
    print(
        f"{col}: "
        f"min={df[col].min()}, "
        f"max={df[col].max()}"
    )

print("\nCATEGORICAL VALUES")
print("=" * 50)

for col in [
    "Gender",
    "Workout_Type",
    "Experience_Level"
]:
    print(f"{col}: {sorted(df[col].dropna().unique().tolist())}")

NUMERIC RANGES
Age: min=18, max=59
Weight (kg): min=40.0, max=129.9
Height (m): min=1.5, max=2.0
Max_BPM: min=160, max=199
Avg_BPM: min=120, max=169
Resting_BPM: min=50, max=74
Session_Duration (hours): min=0.5, max=2.0
Fat_Percentage: min=10.0, max=35.0
Water_Intake (liters): min=1.5, max=3.7
Workout_Frequency (days/week): min=2, max=5
BMI: min=12.32, max=49.84

CATEGORICAL VALUES
Gender: ['Female', 'Male']
Workout_Type: ['Cardio', 'HIIT', 'Strength', 'Yoga']
Experience_Level: [1, 2, 3]
